# ComfyUI Headless Compute Node (First-Principles Minimal Colab Architecture)

- **Core Engine**: `uucaix-jpg/ComfyUI` (`comfy-aimdo` VBAR async host-pinned streaming + `comfy-kitchen` quantized kernels + per-block CUDA Graph compiler)
- **Storage Tiering**: Google Drive cold mirror (`/content/drive/MyDrive/ComfyUI_Store`, sequential copy only) -> `/dev/shm/comfy_models` RAM disk (zero FUSE `mmap` stall, ~57 GB fits **both** `FL2VA` + `Ref2VA` simultaneously) -> `/content/comfy_models` NVMe overflow
- **Gateway Contract**: Idempotent single-cell daemon (`0.0s` warm re-run) + `cloudflared` tunnel + `COLAB_READY_JSON` + automatic `127.0.0.1:18080/api/admin/config` sync

Optional environment overrides before running Cell 1: `VARIANT` (`both` | `fl2va` | `ref2va`), `INCLUDE_TURBO_LORA` (`1` | `0`), `MOUNT_DRIVE` (`0` | `1`), `MODELS_JSON` (custom manifest JSON).

In [ ]:
import json, os, re, shutil, subprocess, sys, time, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# 1. Deterministic, zero-stall transfer & CUDA allocator settings
os.environ.update({
    "HF_HUB_DISABLE_XET": "1",
    "HF_HUB_DISABLE_SHARED_BLOBS": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "PYTHONUNBUFFERED": "1",
})

PORT = int(os.environ.get("PORT", "8188"))
REPO_URL = os.environ.get("REPO_URL", "https://github.com/uucaix-jpg/ComfyUI.git")
REPO_DIR = Path("/content/ComfyUI")
VARIANT = os.environ.get("VARIANT", "both").lower()  # 'fl2va' | 'ref2va' | 'both'
INCLUDE_TURBO_LORA = os.environ.get("INCLUDE_TURBO_LORA", "1") == "1"
MOUNT_DRIVE = os.environ.get("MOUNT_DRIVE", "0") == "1"
SHM_RESERVE_GB = float(os.environ.get("SHM_RESERVE_GB", "16.0"))
STATE_FILE = Path("/content/colab_gateway_state.json")

if MOUNT_DRIVE and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_STORE = DRIVE_ROOT / "ComfyUI_Store" if DRIVE_ROOT.exists() else None
OUTPUT_DIR = (DRIVE_ROOT / "MiniMax-H3-Studio") if DRIVE_ROOT.exists() else Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SHM_MODELS = Path("/dev/shm/comfy_models") if Path("/dev/shm").exists() else Path("/content/comfy_models")
DISK_MODELS = Path("/content/comfy_models")
MODEL_SUBDIRS = ["diffusion_models", "text_encoders", "vae", "loras", "checkpoints", "clip_vision", "controlnet"]
for root in {SHM_MODELS, DISK_MODELS}:
    for sub in MODEL_SUBDIRS:
        (root / sub).mkdir(parents=True, exist_ok=True)

def _http_ok(url: str, timeout: float = 2.0) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

# 2. Ensure uv + huggingface_hub for fast dependency & weight staging
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    subprocess.run(["uv", "pip", "install", "--system", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import hf_hub_download

# 3. Single-file quantized manifest (~37.5 GB single variant / ~57 GB both FL2VA + Ref2VA)
def _build_manifest() -> list[dict]:
    if os.environ.get("MODELS_JSON"):
        return json.loads(os.environ["MODELS_JSON"])
    repo = "Comfy-Org/MiniMax-H3"
    items = [
        {"repo": repo, "path": "split_files/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "subdir": "text_encoders", "est_gb": 14.6},
        {"repo": repo, "path": "split_files/vae/minimax_h3_video_vae_int8_convrot.safetensors", "subdir": "vae", "est_gb": 2.85},
        {"repo": repo, "path": "split_files/vae/minimax_h3_audio_vae_fp32.safetensors", "subdir": "vae", "est_gb": 0.65},
    ]
    if VARIANT in ("fl2va", "both"):
        items.append({"repo": repo, "path": "split_files/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "subdir": "diffusion_models", "est_gb": 19.5})
        if INCLUDE_TURBO_LORA:
            items.append({"repo": repo, "path": "split_files/loras/minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors", "subdir": "loras", "est_gb": 0.8})
    if VARIANT in ("ref2va", "both"):
        items.append({"repo": repo, "path": "split_files/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors", "subdir": "diffusion_models", "est_gb": 19.5})
        if INCLUDE_TURBO_LORA:
            items.append({"repo": repo, "path": "split_files/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", "subdir": "loras", "est_gb": 0.8})
    return items

def _stage_one(item: dict, target_root: Path) -> str:
    fname = Path(item["path"]).name
    subdir = item["subdir"]
    shm_file = SHM_MODELS / subdir / fname
    disk_file = DISK_MODELS / subdir / fname
    if shm_file.exists() and shm_file.stat().st_size > 0:
        return f"hit:/dev/shm/{fname}"
    if disk_file.exists() and disk_file.stat().st_size > 0:
        return f"hit:/content/{fname}"

    dest = target_root / subdir / fname
    cold = (DRIVE_STORE / subdir / fname) if DRIVE_STORE is not None else None
    if cold is not None and cold.exists() and cold.stat().st_size > 0:
        tmp = dest.with_suffix(dest.suffix + ".part")
        shutil.copyfile(cold, tmp)
        tmp.replace(dest)
        return f"drive->{target_root.name}/{fname}"

    hf_token = os.environ.get("HF_TOKEN") or None
    dl_dir = target_root / ".hf_staging"
    dl_path = Path(hf_hub_download(
        repo_id=item["repo"],
        filename=item["path"],
        local_dir=str(dl_dir),
        token=hf_token,
    ))
    shutil.move(str(dl_path), str(dest))
    if cold is not None and os.environ.get("SYNC_DRIVE_STORE", "1") == "1":
        cold.parent.mkdir(parents=True, exist_ok=True)
        cold_tmp = cold.with_suffix(cold.suffix + ".part")
        shutil.copyfile(dest, cold_tmp)
        cold_tmp.replace(cold)
    return f"hf->{target_root.name}/{fname}"

manifest = _build_manifest()
shm_free_gb = shutil.disk_usage(SHM_MODELS).free / (1024 ** 3) if SHM_MODELS != DISK_MODELS else 0.0
shm_budget_gb = max(0.0, shm_free_gb - SHM_RESERVE_GB)
staged_plan = []
for entry in manifest:
    fname = Path(entry["path"]).name
    already = (SHM_MODELS / entry["subdir"] / fname).exists() or (DISK_MODELS / entry["subdir"] / fname).exists()
    est = float(entry.get("est_gb", 1.0))
    if not already and SHM_MODELS != DISK_MODELS and shm_budget_gb >= est:
        shm_budget_gb -= est
        staged_plan.append((entry, SHM_MODELS))
    else:
        staged_plan.append((entry, SHM_MODELS if (SHM_MODELS / entry["subdir"] / fname).exists() else DISK_MODELS))

t0 = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    futs = [pool.submit(_stage_one, item, root) for item, root in staged_plan]
    stage_results = [f.result() for f in as_completed(futs)]
shutil.rmtree(SHM_MODELS / ".hf_staging", ignore_errors=True)
shutil.rmtree(DISK_MODELS / ".hf_staging", ignore_errors=True)
print(f"[weights] staged {len(stage_results)} artifacts in {time.time() - t0:.1f}s: {stage_results}")

# Write unified multi-tier model path config (/dev/shm hot tier + /content overflow tier)
extra_yaml = Path("/content/extra_model_paths.yaml")
yaml_lines = ["colab_shm:", f"  base_path: {SHM_MODELS}"]
for sub in MODEL_SUBDIRS:
    yaml_lines.append(f"  {sub}: {sub}")
if DISK_MODELS != SHM_MODELS:
    yaml_lines.extend(["colab_disk:", f"  base_path: {DISK_MODELS}"])
    for sub in MODEL_SUBDIRS:
        yaml_lines.append(f"  {sub}: {sub}")
extra_yaml.write_text("\n".join(yaml_lines) + "\n", encoding="utf-8")

# 4. Idempotent ComfyUI Core Daemon Boot (0.0s warm reuse if already live)
LOCAL_HEALTH = f"http://127.0.0.1:{PORT}/system_stats"
if _http_ok(LOCAL_HEALTH):
    print(f"[engine] warm ComfyUI daemon already live on 127.0.0.1:{PORT} (0.0s reuse)")
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    reqs = [
        ln.strip() for ln in (REPO_DIR / "requirements.txt").read_text(encoding="utf-8").splitlines()
        if ln.strip() and not ln.strip().startswith("#")
        and re.split(r"[<>=~!\[]", ln.strip())[0].lower() not in {"torch", "torchvision", "torchaudio"}
    ]
    subprocess.run(["uv", "pip", "install", "--system", "-q", *reqs], check=True)

    log_fh = open("/content/comfyui.log", "a", buffering=1)
    subprocess.Popen(
        [
            sys.executable, "main.py",
            "--listen", "127.0.0.1",
            "--port", str(PORT),
            "--extra-model-paths-config", str(extra_yaml),
            "--output-directory", str(OUTPUT_DIR),
            "--database-url", "sqlite:///:memory:",
            "--offline",
            "--disable-auto-launch",
            "--dont-print-server",
        ],
        cwd=str(REPO_DIR),
        stdout=log_fh,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    deadline = time.time() + 120.0
    while time.time() < deadline:
        if _http_ok(LOCAL_HEALTH):
            break
        time.sleep(0.5)
    else:
        tail = "\n".join(Path("/content/comfyui.log").read_text(errors="replace").splitlines()[-40:])
        raise RuntimeError(f"ComfyUI failed to start within 120s:\n{tail}")
    print(f"[engine] ComfyUI compute daemon online at 127.0.0.1:{PORT}")

# 5. Zero-middleman cloudflared tunnel + automatic Studio gateway handshake
tunnel_url = None
if STATE_FILE.exists():
    try:
        prev = json.loads(STATE_FILE.read_text(encoding="utf-8")).get("colab_url", "")
        if prev and _http_ok(f"{prev.rstrip('/')}/system_stats", timeout=4.0):
            tunnel_url = prev.rstrip("/")
    except Exception:
        pass

if not tunnel_url:
    cf_bin = Path("/usr/local/bin/cloudflared")
    if not cf_bin.exists():
        cf_bin = Path("/content/cloudflared")
        if not cf_bin.exists():
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                str(cf_bin),
            )
            cf_bin.chmod(0o755)
    cf_log = Path("/content/cloudflared.log")
    cf_log.write_text("", encoding="utf-8")
    cf_fh = open(cf_log, "a", buffering=1)
    subprocess.Popen(
        [str(cf_bin), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=cf_fh,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    pat = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    deadline = time.time() + 45.0
    while time.time() < deadline:
        m = pat.search(cf_log.read_text(errors="replace"))
        if m:
            tunnel_url = m.group(0)
            break
        time.sleep(0.4)
    if not tunnel_url:
        raise RuntimeError(f"cloudflared did not emit a tunnel URL:\n{cf_log.read_text(errors='replace')}")

with urllib.request.urlopen(LOCAL_HEALTH, timeout=5.0) as resp:
    stats = json.loads(resp.read().decode("utf-8"))
dev0 = (stats.get("devices") or [{}])[0]
payload = {
    "colab_url": tunnel_url,
    "comfy_base_url": tunnel_url,
    "engine": "comfyui",
    "variant": VARIANT,
    "gpu": dev0.get("name", "unknown"),
    "vram_total_gb": round(dev0.get("vram_total", 0) / (1024 ** 3), 2),
    "shm_free_gb": round(shutil.disk_usage(SHM_MODELS).free / (1024 ** 3), 2),
    "output_dir": str(OUTPUT_DIR),
}
STATE_FILE.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

try:
    from IPython.display import Javascript, display
    display(Javascript(
        f"fetch('http://127.0.0.1:18080/api/admin/config', {{"
        f"method: 'POST', headers: {{'Content-Type': 'application/json'}}, "
        f"body: JSON.stringify({json.dumps(payload)})}}).catch(()=>{{}});"
    ))
except Exception:
    pass

print("COLAB_READY_JSON:" + json.dumps(payload, ensure_ascii=False))